# Práctica guiada — De formulario a experiencia de inferencia

En S5 ya construisteis un formulario Streamlit que recoge los 11 atributos y llama a un gateway basado en el contrato de S4. En S6 **no se rediseña el contrato ni se vuelve a implementar el formulario**. El objetivo es convertir esa pantalla funcional en una interfaz operable: mantiene el resultado al haber un rerun, hace visible lo que está ocurriendo y enseña la información justa para interpretar e investigar una predicción.

Trabajad en parejas. Completad los `TODO`, ejecutad las comprobaciones y conservad el notebook como evidencia para el taller. La implementación queda para la Clase 2.

## Entrega de la práctica

Al terminar debéis poder enseñar cuatro decisiones concretas:

1. qué permanece en la sesión, qué se cachea y qué no se retiene;
2. la máquina `idle → loading → success/error`;
3. un plano de pantalla con componentes Streamlit concretos;
4. mensajes seguros para confianza baja, lentitud y error.

No se entrega código de inferencia nuevo. S4 sigue siendo la única frontera de validación y predicción.

## 1. Comprobar el punto de partida de S5

Ejecutad la celda. Debe demostrar que partís del formulario y gateway de S5, no de una segunda implementación. Después anotad una limitación que observaríais al cambiar un widget o al enviar una segunda predicción.

In [6]:
from pathlib import Path

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'semana5/modules/05-streamlit-basic-model-ui').is_dir()
)
s5_app = repo_root / 'semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py'
source = s5_app.read_text(encoding='utf-8')

checks = {
    'reutiliza st.form': 'st.form(' in source,
    'recoge valores antes de inferir': 'collect_values' in source,
    'aún no usa session_state': 'st.session_state' not in source,
}
print('app de S5:', s5_app)
for label, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {label}")
assert all(checks.values())

# TODO: escribid aquí una limitación observable de S5.
limitacion_s5 =  (
    "En la semana 5 podíamos hacer predicciones, pero no guardábamos "
    "los resultados. Esto hace que, cuando Streamlit vuelve a ejecutar "
    "la aplicación, se pierda la última predicción."
)
print('Limitación observada:', limitacion_s5)

app de S5: c:\Users\claud\master\repos\icai-muiaap-operacion-de-modelos1\semana5\modules\05-streamlit-basic-model-ui\solutions\02-first-streamlit\app.py
✓ reutiliza st.form
✓ recoge valores antes de inferir
✓ aún no usa session_state
Limitación observada: En la semana 5 podíamos hacer predicciones, pero no guardábamos los resultados. Esto hace que, cuando Streamlit vuelve a ejecutar la aplicación, se pierda la última predicción.


## 2. Decidir dónde vive cada cosa

Un rerun vuelve a ejecutar el script. Una variable local desaparece; `st.session_state` conserva información de **esa sesión**; `st.cache_resource` reutiliza recursos como un gateway o modelo. Un recurso cacheado global puede compartirse entre usuarios, así que debe ser seguro para concurrencia.

Completad la tabla y justificad cada decisión. No guardéis secretos ni las 11 features en la telemetría. Los valores del último envío solo se conservan para permitir un reintento y no se muestran ni se registran.

| Clave o recurso | Dónde vive | ¿Sobrevive a un rerun? | ¿Se limpia al pulsar «Limpiar»? | Motivo |
| --- | --- | --- | --- | --- |
| `last_state` | st.session_state | si | si | Para guardar el estado de la última predicción y no perder el resultado cada vez que se actualiza la app. |
| `last_values` | st.session_state | si | si | Para poder volver a intentar una predicción sin tener que introducir todos los datos otra vez. |
| `telemetry` | st.session_state | si | no | Para llevar un registro general de las predicciones que hemos hecho, incluso si limpiamos el resultado. |
| gateway / bundle | st.cache_resource | si | no | Para que el modelo no tenga que cargarse de nuevo cada vez que usamos la app. |
| respuesta de una petición | st.cache_resource | si | si | Para seguir mostrando el último resultado sin tener que volver a hacer la predicción. |

Referencias: [`st.session_state`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state) y [`st.cache_resource`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource).

## 3. Diseñar la pantalla, no solo el estado

La interfaz debe tener cuatro zonas, en este orden:

1. **acción:** el formulario heredado de S5;
2. **operación:** qué está ocurriendo mientras se ejecuta;
3. **resultado:** categoría, confianza y latencia;
4. **trazabilidad:** versiones y `request_id`, visibles solo si se necesitan.

Elegid componentes que tengan una función. `st.columns` organiza, `st.status` comunica una operación, `st.metric` destaca un dato y `st.expander` aplaza el detalle técnico. No uséis una barra de progreso para la confianza: representa avance de una tarea, no certeza del modelo.

In [7]:
# Sustituid cada TODO por el componente y la justificación que usaríais.
# Las claves describen zonas de la interfaz; no son nuevos contratos de ML.

ui_plan = {
    'formulario': {
        'component': 'st.form',
        'reason': 'Para introducir los datos del vino y enviar todos los valores juntos al pulsar el botón.'
    },
    'estado_operacion': {
        'component': 'st.status',
        'reason': 'Para que el usuario sepa si la predicción se está ejecutando.'
    },
    'resumen_resultado': {
        'component': 'st.metric',
        'reason': 'Para mostrar de forma clara el resultado, la confianza y la latencia.'
    },
    'mensaje_interpretacion': {
        'component': 'st.warning',
        'reason': 'Para avisar cuando la confianza es baja y recordar que la predicción no es una garantía.'
    },
    'trazabilidad': {
        'component': 'st.expander',
        'reason': 'Para consultar las versiones del modelo y el request_id sin llenar la pantalla de información técnica.'
    },
    'separacion_telemetria': {
        'component': 'st.divider',
        'reason': 'Para separar los datos de telemetría del resultado de la predicción.'
    },
}

allowed_components = {
    'st.form', 'st.status', 'st.empty', 'st.columns', 'st.container',
    'st.metric', 'st.success', 'st.info', 'st.warning', 'st.error',
    'st.expander', 'st.divider', 'st.caption',
}
required_zones = {
    'formulario', 'estado_operacion', 'resumen_resultado',
    'mensaje_interpretacion', 'trazabilidad', 'separacion_telemetria',
}

assert set(ui_plan) == required_zones
for zone, decision in ui_plan.items():
    assert set(decision) == {'component', 'reason'}, zone

pending = [zone for zone, decision in ui_plan.items() if 'TODO' in decision.values()]
invalid = [
    f"{zone}: {decision['component']}"
    for zone, decision in ui_plan.items()
    if 'TODO' not in decision['component'] and decision['component'] not in allowed_components
]
print('Zonas pendientes:', pending or 'ninguna')
print('Componentes no permitidos:', invalid or 'ninguno')
print('Comprobación final: pending == [] and invalid == []')

Zonas pendientes: ninguna
Componentes no permitidos: ninguno
Comprobación final: pending == [] and invalid == []


### Plano mínimo esperado

```text
Título y contexto breve

[ formulario de S5 en dos columnas ]

[ estado de la operación ]

[ Categoría ]  [ Confianza ]  [ Latencia ]
mensaje interpretativo de confianza y latencia
▶ Trazabilidad: versiones y request_id

────────────────────────────────────
Telemetría agregada de la sesión
```

Podéis entregar este plano en el notebook o redibujarlo en draw.io. Justificad cualquier cambio respecto a esta estructura.

## 4. Máquina de estados y recuperación

Completad la tabla. Una respuesta válida con latencia superior a 300 ms sigue en `success`: añade un aviso técnico, no se convierte en error. La confianza es una señal para revisar, no un diagnóstico ni una garantía.

| Estado | Evento de entrada | Zona de pantalla que cambia | Qué se muestra | Acción permitida | Evidencia |
| --- | --- | --- | --- | --- | --- |
| `idle` | Inicio / limpiar | Zona de resultados | Un mensaje indicando que hay que rellenar el formulario para hacer una predicción. | Introducir datos y enviar el formulario. | Aparece el mensaje inicial y no hay ninguna predicción. |
| `loading` | Enviar formulario | Estado de la operacion | Un mensaje indicando que el modelo está procesando los datos. | Esperar a que termine la predicción. | Se muestra el estado de carga. |
| `success` | Payload válido | Zona de resultados | La categoría, la confianza y la latencia. Si la confianza es baja o tarda más de 300 ms, aparece un aviso. | Hacer otra predicción o limpiar el resultado. | Se muestran los resultados y los avisos correspondientes. |
| `error` | Excepción controlada | Zona de resultados y errores | Un mensaje explicando qué ha fallado y cómo solucionarlo, junto con el request_id. | Corregir los datos, reintentar o limpiar. | Aparece el mensaje de error sin mostrar información interna. |

Incluid una recuperación para entrada inválida, bundle no disponible y timeout. Indicad también qué no debe ver la persona: traceback, rutas internas, secretos o el payload completo.

Entrada inválida: mostramos un mensaje para que el usuario revise los valores introducidos y vuelva a enviar el formulario.

Bundle no disponible: avisamos de que el modelo no está disponible y que hay que revisar la configuración antes de volver a intentarlo.

Timeout: indicamos que la predicción ha tardado demasiado y damos la opción de reintentar.

Limpiar: volvemos al estado idle y eliminamos el último resultado, pero mantenemos la telemetría de la sesión.

## 5. Escribir el copy de una interfaz responsable

Cada mensaje debe decir qué ha ocurrido, qué significa y qué puede hacer la persona. No debéis prometer que una predicción es correcta. Completad textos de una frase, sin terminología interna.

In [9]:
ux_copy = {
    'low_confidence': 'La predicción tiene una confianza baja, por lo que recomendamos revisar el resultado antes de sacar conclusiones.',
    'slow_success': 'La predicción se ha completado correctamente, aunque ha tardado más de lo esperado.',
    'invalid_input': 'Algunos datos introducidos no son válidos, revisa los campos y vuelve a intentarlo.',
    'artifact_unavailable': 'No podemos realizar la predicción porque el modelo no está disponible en este momento, inténtalo de nuevo más tarde.',
}

assert set(ux_copy) == {
    'low_confidence', 'slow_success', 'invalid_input', 'artifact_unavailable'
}
for scenario, message in ux_copy.items():
    assert isinstance(message, str) and message.strip(), scenario

pending_copy = [scenario for scenario, message in ux_copy.items() if 'TODO' in message]
print('Mensajes pendientes:', pending_copy or 'ninguno')
print('Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?')

Mensajes pendientes: ninguno
Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?


## 6. Preparar el taller

Entregad el notebook con los TODO completados. Antes de cerrar, otra pareja debe poder responder estas preguntas leyendo vuestro trabajo:

- ¿Qué se mantiene al pulsar otro widget y qué se elimina con «Limpiar último resultado»?
Se mantienen el último resultado y la telemetría. Al pulsar «Limpiar», borramos el último resultado y los valores guardados para reintentar, pero no la telemetría.

- ¿Qué función cargará el gateway una sola vez y cuál será su límite?
build_gateway(), utilizando st.cache_resource para reutilizar el modelo sin cargarlo en cada rerun. La caché guarda el gateway, no las predicciones.

- ¿Qué componentes dibujará `render_state()` para `loading`, `success` y `error`?
En loading, un mensaje de carga con st.status; en success, las métricas y los avisos; y en error, un mensaje explicando el problema y cómo solucionarlo.

- ¿Qué ve una persona con confianza baja o una respuesta lenta?
Mostramos una advertencia. Si la respuesta tarda más de 300 ms, sigue siendo un resultado válido y no un error.

- ¿Dónde aparecen el `request_id` y las versiones sin invadir el resultado?
Dentro de un st.expander llamado «Trazabilidad», para que estén disponibles sin ocupar espacio en el resultado principal.

En el taller se implementarán estas decisiones en `session.py`, `policies.py`, `presentation.py`, `controller.py` y `app.py`.

## Debrief

- ¿Qué limita a S5 aunque el formulario ya funcione?
En S5 podemos introducir datos y obtener una predicción, pero no gestionamos el estado de la aplicación. Por eso, al producirse un rerun podemos perder el último resultado y no tenemos opciones para reintentar o consultar las predicciones anteriores.

- ¿Por qué el gateway es un recurso cacheable y el resultado de una petición pertenece a una sesión?
Porque el gateway utiliza el mismo modelo para hacer las predicciones, así que podemos guardarlo con st.cache_resource y evitar cargarlo varias veces. En cambio, cada resultado depende de los datos introducidos por el usuario, por lo que debe guardarse en st.session_state.

- ¿Qué componente hace visible cada transición sin añadir una segunda lógica de inferencia?
Podemos utilizar st.status para mostrar cuándo se está ejecutando una predicción, st.success para indicar que ha terminado correctamente y st.error cuando ocurre algún problema. Estos componentes solo muestran información, mientras que el gateway sigue encargándose de la inferencia.

- ¿Por qué categoría, confianza y latencia no tienen el mismo significado?
Porque cada una nos da información diferente. La categoría es el resultado que predice el modelo, la confianza indica cómo de seguro está el modelo de esa predicción y la latencia es el tiempo que ha tardado en responder. Una confianza alta no garantiza que la predicción sea correcta y una latencia elevada no significa que sea incorrecta.

- ¿Qué parte de este diseño se conserva al sustituir el gateway local por HTTP en S7?
Podemos mantener el formulario, los estados de la aplicación, los mensajes y la forma de mostrar los resultados. Lo que cambia es el gateway, que en lugar de ejecutar el modelo localmente hará una petición HTTP para obtener la predicción.
